Task B: DPO

Prasanna Paithankar (21CS30065)

Hugging Face Hub Links:
- Full Fine-tuning: PrasannaPaithankar/gpt2-medium-dpo-full
- Prefix Tuning: PrasannaPaithankar/gpt2-medium-dpo-prefix
- LoRA: PrasannaPaithankar/gpt2-medium-dpo-lora
- QLoRA: PrasannaPaithankar/gpt2-medium-dpo-qlora

In [1]:
!pip install -q trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 10.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
from trl import DPOTrainer

In [ ]:
import gc
import os

import torch
import torch.nn.functional as F
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, PrefixTuningConfig, TaskType, get_peft_model
from rich.progress import (
    BarColumn,
    Progress,
    TaskProgressColumn,
    TextColumn,
    TimeRemainingColumn,
)
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

In [ ]:
ds = load_dataset("nrizwan/safe_ai_assignment_1")
ds["train"] = ds["train"].shuffle(seed=42).select(range(4000))

model_name = "gpt2-medium"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def collate_dpo(batch):
    prompts = [item["Question"] for item in batch]
    chosen = [item["More_Prefered"] for item in batch]
    rejected = [item["Less_Prefered"] for item in batch]
    return prompts, chosen, rejected


dataloader = DataLoader(ds["train"], batch_size=4, collate_fn=collate_dpo)

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

preference_train.csv:   0%|          | 0.00/81.5M [00:00<?, ?B/s]

preference_test.csv:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
def setup(strategy):
    if strategy == "qlora":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32)
        return get_peft_model(model, peft_config)
    elif strategy == "lora":
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )
        peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32)
        return get_peft_model(model, peft_config)
    elif strategy == "prefix":
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )
        peft_config = PrefixTuningConfig(
            task_type=TaskType.CAUSAL_LM, num_virtual_tokens=20
        )
        return get_peft_model(model, peft_config)
    else:
        return AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )

In [ ]:
def batch_log_ps(logits, labels):
    slogs = logits[..., :-1, :].contiguous()
    slabels = labels[..., 1:].contiguous()
    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
    token_logps = -loss_fct(slogs.view(-1, slogs.size(-1)), slabels.view(-1))
    token_logps = token_logps.view(slabels.size())
    mask = slabels != tokenizer.pad_token_id
    return (token_logps * mask).sum(-1)


def log_ps(model, prompts, responses):
    inputs = [p + r for p, r in zip(prompts, responses)]
    encodings = tokenizer(
        inputs, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(model.device)
    prompt_encodings = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(model.device)

    prompt_lengths = (prompt_encodings.input_ids != tokenizer.pad_token_id).sum(dim=-1)

    labels = encodings.input_ids.clone()
    for i, length in enumerate(prompt_lengths):
        labels[i, :length] = tokenizer.pad_token_id

    with torch.amp.autocast("cuda", dtype=torch.float16):
        logits = model(**encodings).logits

    labels = labels.to(logits.device)

    return batch_log_ps(logits, labels)


def dpo_loss(
    pi_chosen_logps, pi_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1
):
    target_device = pi_chosen_logps.device

    pi_rejected_logps = pi_rejected_logps.to(target_device)
    ref_chosen_logps = ref_chosen_logps.to(target_device)
    ref_rejected_logps = ref_rejected_logps.to(target_device)

    pi_logratios = pi_chosen_logps - pi_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    logits = pi_logratios - ref_logratios
    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

In [ ]:
def dpo_loop(strategy, epochs=1):
    policy_model = setup(strategy)
    ref_model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map="auto"
    ).eval()
    for param in ref_model.parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-5)

    total_steps = len(dataloader) * epochs
    step_losses = []
    with Progress(
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
        TimeRemainingColumn(),
    ) as progress:
        task = progress.add_task(
            f"[cyan]Training {strategy.upper()}...", total=total_steps
        )

        for epoch in range(epochs):
            for step, (prompts, chosen, rejected) in enumerate(dataloader):
                optimizer.zero_grad()

                with torch.no_grad():
                    ref_chosen_logps = log_ps(ref_model, prompts, chosen)
                    ref_rejected_logps = log_ps(ref_model, prompts, rejected)

                pi_chosen_logps = log_ps(policy_model, prompts, chosen)
                pi_rejected_logps = log_ps(policy_model, prompts, rejected)

                loss = dpo_loss(
                    pi_chosen_logps,
                    pi_rejected_logps,
                    ref_chosen_logps,
                    ref_rejected_logps,
                )

                loss.backward()
                optimizer.step()

                current_loss = loss.item()
                step_losses.append(current_loss)
                progress.update(
                    task,
                    advance=1,
                    description=f"[cyan]{strategy.upper()} | Epoch {epoch + 1}/{epochs} | Loss: {loss.item():.4f}",
                )

    policy_model.push_to_hub(f"PrasannaPaithankar/gpt2-medium-dpo-{strategy}")

    del policy_model
    del ref_model
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()
    return step_losses

In [ ]:
all_strategy_losses = {}

for strat in ["qlora", "lora", "prefix", "full"]:
    all_strategy_losses[strat] = dpo_loop(strat)

import pickle

with open("losses.pkl", "wb") as file:
    pickle.dump(all_strategy_losses, file)

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Output()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            